# Modelo THANOS: Prediccion Electoral Colombia 2026

Implementacion del plan estrategico reorganizado (Fases 0, 1a, 2, 3, 1b).
Cada seccion produce datos intermedios guardados en `data/processed/thanos_*.parquet`.

Orden de ejecucion:
- **Fase 0**: Exprimir datos existentes (E1-E7, costo \$0)
- **Fase 1a**: Monitoreo inmediato de picos y share of voice (E9-E10, arranque inmediato)
- **Fase 2**: Descarga quirurgica de tweets ciudadanos (E11-E15, \$30-\$80)
- **Fase 3**: Expansion de red via listas curadas en X (E16-E17, \$50-\$150)
- **Fase 1b**: Tracking de hashtags con diccionario enriquecido (E8, \$3-\$7/mes)

Referencia: `docs/plan_estrategico_obtencion_informacion_tweeter.md`

In [1]:
import sys
sys.path.insert(0, "..")

import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import networkx as nx
import numpy as np
import statsmodels.api as sm
from datetime import date
import glob as _glob

from src.config import DATA_RAW, DATA_PROCESSED, PROJECT_ROOT
from src.twitter_client import get_client
from src.cost_estimator import estimate_search, estimate_timeline, estimate_user_lookup, print_estimate
from src.collectors.tweet_collector import search_tweets, collect_user_timeline, save_tweets, SINCE_DATE

import thanos_helpers as th
from importlib import reload
reload(th)

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(80)

polars.config.Config

### Carga de datos base

In [2]:
# Cargar todos los datos existentes
df_all = th.load_all_seed_tweets()
candidates = th.load_candidates_config()
edges = th.load_edges()
encuestas = th.load_encuestas()
seed_handles = th.get_all_seed_handles(candidates)

print(f"Tweets semilla: {len(df_all):,}")
print(f"  Derecha: {df_all.filter(pl.col('bloque') == 'derecha').height:,}")
print(f"  Izquierda: {df_all.filter(pl.col('bloque') == 'izquierda').height:,}")
print(f"Edges: {len(edges):,}")
print(f"Encuestas: {len(encuestas)}")
print(f"Seed handles: {seed_handles}")

Tweets semilla: 2,373
  Derecha: 1,412
  Izquierda: 961
Edges: 1,849
Encuestas: 7
Seed handles: {'ivancepedacast', 'cedemocratico', 'abdelaespriella', 'petrogustavo', 'pactohistorico', 'pizarromariajo', 'palomavalencial', 'alvarouribevel', 'gustavobolivar', 'jdoviedoar'}


In [3]:
df_all.head()

tweet_id,author_id,author_username,author_name,author_created_at,author_followers,author_following,author_tweet_count,author_verified,author_description,author_location,text,created_at,lang,source,conversation_id,in_reply_to_user_id,ref_type,ref_tweet_id,retweet_count,reply_count,like_count,quote_count,impression_count,hashtags,mentions,collected_at,bloque
str,str,str,str,str,i64,i64,i64,bool,str,str,str,str,str,null,str,str,str,str,i64,i64,i64,i64,i64,list[str],list[str],str,str
"""2033546318887506037""","""219434063""","""JDOviedoAr""","""Juan Daniel Oviedo""","""2010-11-24T21:21:15.000Z""",167168,9075,66220,false,"""Aspirante a la Vicepresidencia. Somos el centro que sí se moja. Sumemos, aún en …","""Bogotá, D.C., Colombia""","""RT @DiversaBogota: No hay cabida para la homofobia, es más no lo podemos permiti…","""2026-03-16T14:09:58.000Z""","""es""",null,"""2033546318887506037""",null,"""retweeted""","""2033244695313232227""",1118,0,0,0,14,[],"[""DiversaBogota""]","""2026-04-03T20:52:11.746124""","""derecha"""
"""2038339881361875209""","""1115440213""","""CeDemocratico""","""Centro Democrático""","""2013-01-23T22:30:02.000Z""",519687,435,245471,false,"""Partido Político - Centro Democrático. Mano Firme, Corazón Grande. @legadouribev…","""Colombia""","""RT @PalomaValenciaL: Con toda la devoción y el compromiso de trabajar por un paí…","""2026-03-29T19:37:52.000Z""","""es""",null,"""2038339881361875209""",null,"""retweeted""","""2038316134852923470""",405,0,0,0,0,[],"[""PalomaValenciaL""]","""2026-04-03T20:52:11.746124""","""derecha"""
"""2040068589382676970""","""149281495""","""PalomaValenciaL""","""Paloma Valencia L""","""2010-05-28T22:14:20.000Z""",641065,1394,51985,true,"""Candidata a la Presidencia de la Gran Consulta por Colombia. El 31 de mayo vot…","""Bogotá, Colombia""","""RT @PalomaValenciaL: Así se vive la Semana Santa en Popayán. La procesión de la …","""2026-04-03T14:07:09.000Z""","""es""",null,"""2040068589382676970""",null,"""retweeted""","""2040065274402001172""",139,0,0,0,0,[],"[""PalomaValenciaL""]","""2026-04-03T20:52:11.746124""","""derecha"""
"""2037637397999923435""","""1657059045956517897""","""ABDELAESPRIELLA""","""Abelardo De La Espriella""","""2023-05-12T16:24:25.000Z""",166543,99,4364,false,"""Candidato a la Presidencia de Colombia por el movimiento Defensores de la Patria…",null,"""RT @GermnAndrs92683: COLOMBIA 🇨🇴🚨 Abelardo de la Espriella y la reserva estamos…","""2026-03-27T21:06:27.000Z""","""es""",null,"""2037637397999923435""",null,"""retweeted""","""2037625727726547040""",1281,0,0,0,1,[],"[""GermnAndrs92683"", ""ABDELAESPRIELLA""]","""2026-04-03T20:52:11.746124""","""derecha"""
"""2039016939402518917""","""1362851972""","""PizarroMariaJo""","""María José Pizarro Rodríguez""","""2013-04-18T20:33:56.000Z""",678161,1691,25890,false,"""Senadora de la República - Copresidenta del Pacto Histórico - Jefa de Debate de …","""Bogotá, D.C., Colombia""","""@BecerraGabo Gracias querido @BecerraGabo, por tu mensaje, amistad y trabajo con…","""2026-03-31T16:28:16.000Z""","""es""",null,"""2038832470716567948""","""282188079""","""replied_to""","""2038832470716567948""",3,0,26,0,880,[],"[""BecerraGabo"", ""BecerraGabo""]","""2026-04-03T20:57:40.529941""","""izquierda"""


In [4]:
# Paths y helpers de cache
THANOS_DIR = DATA_PROCESSED

def thanos_path(name: str):
    return THANOS_DIR / f"thanos_{name}.parquet"

def cached(name: str) -> bool:
    return thanos_path(name).exists()

## Fase 0: Exprimir datos existentes (costo $0)

Estrategias E1-E7 trabajan exclusivamente con los datos ya recolectados.
No se hacen llamadas al API.

### E1. Diccionario de hashtags por bloque politico

Extraer todos los hashtags de los tweets semilla, agrupar por bloque, rankear por frecuencia.
Los top 10 por bloque son la entrada directa para `x_{t,h,l}^(k)` de THANOS.

In [5]:
hashtag_dict = th.hashtags_by_bloc(df_all)
hashtag_dict.write_parquet(thanos_path("hashtag_dict"))

print("Top 15 por bloque:")
for bloque in ("derecha", "izquierda"):
    print(f"\n--- {bloque.upper()} ---")
    print(hashtag_dict.filter(pl.col("bloque") == bloque).head(15))

Top 15 por bloque:

--- DERECHA ---
shape: (15, 4)
┌──────────────────────┬─────────┬───────┬──────┐
│ hashtag              ┆ bloque  ┆ count ┆ rank │
│ ---                  ┆ ---     ┆ ---   ┆ ---  │
│ str                  ┆ str     ┆ u32   ┆ u32  │
╞══════════════════════╪═════════╪═══════╪══════╡
│ atención             ┆ derecha ┆ 18    ┆ 1    │
│ política             ┆ derecha ┆ 10    ┆ 2    │
│ periodicazo          ┆ derecha ┆ 9     ┆ 3    │
│ colombiaconpaloma    ┆ derecha ┆ 8     ┆ 4    │
│ colombiamásgrande    ┆ derecha ┆ 7     ┆ 5    │
│ elgranreto2026       ┆ derecha ┆ 7     ┆ 6    │
│ juntosporcolombia    ┆ derecha ┆ 6     ┆ 7    │
│ palovieta            ┆ derecha ┆ 5     ┆ 8    │
│ palomapresidente     ┆ derecha ┆ 4     ┆ 9    │
│ ordenfirmezaycorazón ┆ derecha ┆ 4     ┆ 10   │
│ 10minutoscon         ┆ derecha ┆ 3     ┆ 11   │
│ palomayoviedo        ┆ derecha ┆ 2     ┆ 12   │
│ tourperiodicazo      ┆ derecha ┆ 2     ┆ 13   │
│ confidenciales       ┆ derecha ┆ 2     ┆ 14   │

In [6]:
# Visualizacion: top 15 hashtags por bloque
top_viz = pl.concat([
    hashtag_dict.filter(pl.col("bloque") == b).head(15)
    for b in ("derecha", "izquierda")
])

fig = px.bar(
    top_viz.to_pandas(),
    x="hashtag", y="count", color="bloque",
    barmode="group",
    title="Top 15 hashtags por bloque politico",
    color_discrete_map={"derecha": "#1f77b4", "izquierda": "#d62728"},
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

### E2. Cadenas de retweet y r_t^(k)

Reconstruir las cadenas de retweet desde `ref_type == "retweeted"` y `ref_tweet_id`.
Calcular la proporcion de retweets del influenciador principal por partido.

**Input THANOS:** `r_t^(k)` directo.

In [7]:
rt_chains = th.retweet_chains(df_all, candidates)
rt_chains.write_parquet(thanos_path("retweet_chains"))

print("Cadenas de retweet por bloque:")
print(rt_chains)

Cadenas de retweet por bloque:
shape: (2, 5)
┌───────────┬─────────────────┬───────────────┬────────────────┬──────────┐
│ bloque    ┆ top_influencer  ┆ retweet_count ┆ total_retweets ┆ r_t_k    │
│ ---       ┆ ---             ┆ ---           ┆ ---            ┆ ---      │
│ str       ┆ str             ┆ u32           ┆ u32            ┆ f64      │
╞═══════════╪═════════════════╪═══════════════╪════════════════╪══════════╡
│ derecha   ┆ palomavalencial ┆ 96            ┆ 218            ┆ 0.440367 │
│ izquierda ┆ petrogustavo    ┆ 49            ┆ 134            ┆ 0.365672 │
└───────────┴─────────────────┴───────────────┴────────────────┴──────────┘


In [8]:
# Visualizacion: proporcion de retweets del top influencer por bloque
fig = px.bar(
    rt_chains.to_pandas(),
    x="bloque", y="r_t_k", color="bloque",
    text="top_influencer",
    title="r_t^(k): Proporcion de retweets del influenciador principal",
    color_discrete_map={"derecha": "#1f77b4", "izquierda": "#d62728"},
)
fig.update_traces(textposition="inside")
fig.show()

### E3. Proxy de influencia via author_followers

Usar el maximo de `author_followers` por cuenta para rankear influencia
sin necesidad de hacer lookups de usuario adicionales ($0.010 cada uno).

In [9]:
influence = th.proxy_influence(df_all)
influence.write_parquet(thanos_path("influence_proxy"))

print("Top 20 cuentas por seguidores:")
print(influence.head(20))

Top 20 cuentas por seguidores:
shape: (10, 3)
┌─────────────────┬───────────┬───────────────┐
│ author_username ┆ bloque    ┆ max_followers │
│ ---             ┆ ---       ┆ ---           │
│ str             ┆ str       ┆ i64           │
╞═════════════════╪═══════════╪═══════════════╡
│ petrogustavo    ┆ izquierda ┆ 8366468       │
│ AlvaroUribeVel  ┆ derecha   ┆ 4967766       │
│ IvanCepedaCast  ┆ izquierda ┆ 1933160       │
│ GustavoBolivar  ┆ izquierda ┆ 1724627       │
│ PizarroMariaJo  ┆ izquierda ┆ 678161        │
│ PalomaValenciaL ┆ derecha   ┆ 641065        │
│ CeDemocratico   ┆ derecha   ┆ 519687        │
│ JDOviedoAr      ┆ derecha   ┆ 167168        │
│ ABDELAESPRIELLA ┆ derecha   ┆ 166543        │
│ PactoHistorico  ┆ izquierda ┆ 28497         │
└─────────────────┴───────────┴───────────────┘


### E4. Clustering de hashtags por co-ocurrencia

Construir una matriz de co-ocurrencia: que hashtags aparecen juntos en el mismo tweet.
Hashtags que co-ocurren frecuentemente pertenecen al mismo bloque.
Valida y expande el diccionario de E1 automaticamente.

In [10]:
cooccurrence = th.hashtag_cooccurrence(df_all, min_count=2)
cooccurrence.write_parquet(thanos_path("hashtag_cooccurrence"))

print(f"Pares de co-ocurrencia encontrados: {len(cooccurrence)}")
print(cooccurrence.head(20))

Pares de co-ocurrencia encontrados: 0
shape: (0, 3)
┌───────────┬───────────┬──────────────┐
│ hashtag_a ┆ hashtag_b ┆ cooccurrence │
│ ---       ┆ ---       ┆ ---          │
│ str       ┆ str       ┆ u32          │
╞═══════════╪═══════════╪══════════════╡
└───────────┴───────────┴──────────────┘


In [11]:
# Visualizacion: grafo de co-ocurrencia de hashtags
if len(cooccurrence) > 0:
    G_ht = nx.Graph()
    for row in cooccurrence.head(50).iter_rows(named=True):
        G_ht.add_edge(row["hashtag_a"], row["hashtag_b"], weight=row["cooccurrence"])

    # Colorear por bloque usando diccionario E1
    bloc_map = dict(zip(
        hashtag_dict["hashtag"].to_list(),
        hashtag_dict["bloque"].to_list(),
    ))
    color_map = {"derecha": "#1f77b4", "izquierda": "#d62728", "ambos": "#7f7f7f"}

    pos = nx.spring_layout(G_ht, seed=42, k=2)
    edge_x, edge_y = [], []
    for u, v in G_ht.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    node_x = [pos[n][0] for n in G_ht.nodes()]
    node_y = [pos[n][1] for n in G_ht.nodes()]
    node_colors = [color_map.get(bloc_map.get(n, "ambos"), "#7f7f7f") for n in G_ht.nodes()]
    node_text = list(G_ht.nodes())

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines",
                             line=dict(width=0.5, color="#888"), hoverinfo="none"))
    fig.add_trace(go.Scatter(x=node_x, y=node_y, mode="markers+text",
                             marker=dict(size=10, color=node_colors),
                             text=node_text, textposition="top center", textfont_size=8))
    fig.update_layout(title="Co-ocurrencia de hashtags (top 50 pares)",
                      showlegend=False, xaxis=dict(visible=False), yaxis=dict(visible=False))
    fig.show()
else:
    print("No hay suficientes co-ocurrencias para graficar")

No hay suficientes co-ocurrencias para graficar


### E5. Filtro de bots por heuristicas

Filtrar cuentas probablemente automatizadas:
- Creadas despues de enero 2026
- tweet_count > 10,000
- Ratio following/followers > 10

Limpiar antes de calcular metricas.

In [12]:
bot_flags = th.detect_bots(df_all)
bot_flags.write_parquet(thanos_path("bot_flags"))

bots = bot_flags.filter(pl.col("is_bot"))
print(f"Bots detectados: {len(bots)} de {len(bot_flags)} cuentas unicas")
if len(bots) > 0:
    print(bots)
else:
    print("Ninguna cuenta cumple todos los criterios de bot simultaneamente")
    # Mostrar cuentas que cumplen al menos un criterio
    suspicious = bot_flags.filter(
        pl.col("recent_account") | pl.col("high_volume") | pl.col("high_ratio")
    )
    if len(suspicious) > 0:
        print(f"\nCuentas sospechosas (al menos 1 criterio): {len(suspicious)}")
        print(suspicious)

Bots detectados: 0 de 10 cuentas unicas
Ninguna cuenta cumple todos los criterios de bot simultaneamente

Cuentas sospechosas (al menos 1 criterio): 9
shape: (9, 10)
┌────────────┬────────────┬────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ author_use ┆ author_id  ┆ is_bot ┆ recent_ac ┆ … ┆ author_cr ┆ author_tw ┆ author_fo ┆ author_fo │
│ rname      ┆ ---        ┆ ---    ┆ count     ┆   ┆ eated_at  ┆ eet_count ┆ llowers   ┆ llowing   │
│ ---        ┆ str        ┆ bool   ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆            ┆        ┆ bool      ┆   ┆ str       ┆ i64       ┆ i64       ┆ i64       │
╞════════════╪════════════╪════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ AlvaroUrib ┆ 61097151   ┆ false  ┆ false     ┆ … ┆ 2009-07-2 ┆ 99088     ┆ 4967766   ┆ 7313      │
│ eVel       ┆            ┆        ┆           ┆   ┆ 9T03:08:1 ┆           ┆           ┆           │
│            ┆            

### E6. Ratio replies/likes como proxy de controversia

`reply_count / like_count` alto indica contenido polarizante.
Clasificar tweets por controversia permite ponderar las ventanas temporales:
dias con alta controversia tienen mas senal predictiva.

In [13]:
controversy = th.controversy_ratio(df_all)
controversy.write_parquet(thanos_path("controversy"))

print("Top 10 tweets mas controversiales:")
print(
    controversy
    .select("author_username", "text", "reply_count", "like_count", "controversy_score")
    .head(10)
)

Top 10 tweets mas controversiales:
shape: (10, 5)
┌─────────────────┬─────────────────────────────────┬─────────────┬────────────┬───────────────────┐
│ author_username ┆ text                            ┆ reply_count ┆ like_count ┆ controversy_score │
│ ---             ┆ ---                             ┆ ---         ┆ ---        ┆ ---               │
│ str             ┆ str                             ┆ i64         ┆ i64        ┆ f64               │
╞═════════════════╪═════════════════════════════════╪═════════════╪════════════╪═══════════════════╡
│ PalomaValenciaL ┆ Respetamos las decisiones de la ┆ 1760        ┆ 1220       ┆ 1.442623          │
│                 ┆ justicia. Acompañaremos con     ┆             ┆            ┆                   │
│                 ┆ todo empeño la lucha…           ┆             ┆            ┆                   │
│ PalomaValenciaL ┆ Estados Unidos, Israel y los    ┆ 1836        ┆ 2227       ┆ 0.824427          │
│                 ┆ países árabes le hace

In [14]:
# Scatter: reply_count vs like_count coloreado por bloque
originals = df_all.filter(pl.col("ref_type").is_null()).drop_nulls(["reply_count", "like_count"])

fig = px.scatter(
    originals.to_pandas(),
    x="like_count", y="reply_count", color="bloque",
    hover_data=["author_username", "text"],
    title="Controversia: replies vs likes (tweets originales)",
    color_discrete_map={"derecha": "#1f77b4", "izquierda": "#d62728"},
    log_x=True, log_y=True,
)
# Linea de referencia: ratio = 1
max_val = max(originals["like_count"].max(), originals["reply_count"].max())
fig.add_shape(type="line", x0=1, y0=1, x1=max_val, y1=max_val,
              line=dict(dash="dash", color="gray"))
fig.show()

### E7. Estimacion de replies disponibles

Sumar `reply_count` de todos los tweets de candidatos ya en disco.
Esto da el volumen exacto de tweets ciudadanos disponibles para descargar
y permite presupuestar las estrategias E12 (top hilos) y E14 (menciones).

In [15]:
replies_vol = th.estimate_replies_volume(df_all, seed_handles)

total_replies = replies_vol["total_replies"].sum()
total_cost = replies_vol["estimated_cost_usd"].sum()

print(f"Replies disponibles para descarga: {total_replies:,}")
print(f"Costo estimado de descargar todos: ${total_cost:.2f}")
print(f"\nDesglose por cuenta semilla:")
print(replies_vol)

Replies disponibles para descarga: 159,283
Costo estimado de descargar todos: $796.42

Desglose por cuenta semilla:
shape: (8, 3)
┌─────────────────┬───────────────┬────────────────────┐
│ author_username ┆ total_replies ┆ estimated_cost_usd │
│ ---             ┆ ---           ┆ ---                │
│ str             ┆ i64           ┆ f64                │
╞═════════════════╪═══════════════╪════════════════════╡
│ PalomaValenciaL ┆ 62024         ┆ 310.12             │
│ JDOviedoAr      ┆ 29002         ┆ 145.01             │
│ IvanCepedaCast  ┆ 20522         ┆ 102.61             │
│ GustavoBolivar  ┆ 19156         ┆ 95.78              │
│ PizarroMariaJo  ┆ 10320         ┆ 51.6               │
│ ABDELAESPRIELLA ┆ 9748          ┆ 48.74              │
│ CeDemocratico   ┆ 5996          ┆ 29.98              │
│ PactoHistorico  ┆ 2515          ┆ 12.575             │
└─────────────────┴───────────────┴────────────────────┘


### Checkpoint Fase 0

Verificar que todos los outputs de Fase 0 se generaron correctamente.

In [16]:
fase_0_outputs = [
    "hashtag_dict", "retweet_chains", "influence_proxy",
    "hashtag_cooccurrence", "bot_flags", "controversy",
]
for name in fase_0_outputs:
    path = thanos_path(name)
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    status = "OK" if exists else "FALTA"
    print(f"  [{status}] thanos_{name}.parquet ({size:,} bytes)")

print(f"\nFase 0 completa: {all(thanos_path(n).exists() for n in fase_0_outputs)}")

  [OK] thanos_hashtag_dict.parquet (2,956 bytes)
  [OK] thanos_retweet_chains.parquet (2,000 bytes)
  [OK] thanos_influence_proxy.parquet (1,550 bytes)
  [OK] thanos_hashtag_cooccurrence.parquet (453 bytes)
  [OK] thanos_bot_flags.parquet (4,258 bytes)
  [OK] thanos_controversy.parquet (130,909 bytes)

Fase 0 completa: True


## Fase 1a: Monitoreo inmediato (~$2/mes)

Estrategias E9-E10 no dependen del diccionario de hashtags y se arrancan de inmediato
para acumular serie temporal desde el primer dia.

E8 (tracking de hashtags) se difiere a Fase 1b, despues de enriquecer el diccionario
con hashtags organicos y de coyuntura descubiertos en Fases 2-3.

**IMPORTANTE:** Las celdas de ejecucion estan comentadas para prevenir gasto accidental.
Descomentar manualmente para ejecutar.

### E9. Deteccion de picos noticiosos via count_recent

Query general de elecciones para detectar picos de actividad.
Los dias pico tienen mas senal predictiva.

Costo: $0.005/ejecucion.

In [17]:
# Descomenta para ejecutar (costo: $0.005)
today = date.today().isoformat()
news_counts = th.run_count_recent(["elecciones Colombia 2026 lang:es"])
news_counts.write_parquet(thanos_path(f"news_peaks_{today}"))
print(news_counts)

# Detectar picos: dias con volumen > 2x la mediana
median_vol = news_counts["count"].median()
peaks = news_counts.filter(pl.col("count") > 2 * median_vol)
print(f"\nPicos detectados (>{2 * median_vol:.0f} tweets):")
print(peaks)

Ejecutando 1 queries count_recent (costo: $0.005)
shape: (8, 3)
┌──────────────────────────────────┬────────────┬───────┐
│ query                            ┆ date       ┆ count │
│ ---                              ┆ ---        ┆ ---   │
│ str                              ┆ str        ┆ i64   │
╞══════════════════════════════════╪════════════╪═══════╡
│ elecciones Colombia 2026 lang:es ┆ 2026-03-29 ┆ 39    │
│ elecciones Colombia 2026 lang:es ┆ 2026-03-30 ┆ 362   │
│ elecciones Colombia 2026 lang:es ┆ 2026-03-31 ┆ 504   │
│ elecciones Colombia 2026 lang:es ┆ 2026-04-01 ┆ 704   │
│ elecciones Colombia 2026 lang:es ┆ 2026-04-02 ┆ 416   │
│ elecciones Colombia 2026 lang:es ┆ 2026-04-03 ┆ 278   │
│ elecciones Colombia 2026 lang:es ┆ 2026-04-04 ┆ 595   │
│ elecciones Colombia 2026 lang:es ┆ 2026-04-05 ┆ 1765  │
└──────────────────────────────────┴────────────┴───────┘

Picos detectados (>920 tweets):
shape: (1, 3)
┌──────────────────────────────────┬────────────┬───────┐
│ query            

### E10. Share of voice por candidato

`count_recent` para cada candidato por separado.
La proporcion de menciones entre candidatos es una variable predictiva adicional.

Costo: N candidatos x $0.005/ejecucion.

In [18]:
candidate_queries = th.build_candidate_queries(candidates)
print(f"Queries de share of voice: {len(candidate_queries)}")
print(f"Costo por ejecucion: ${len(candidate_queries) * 0.005:.3f}")
for q in candidate_queries:
    print(f"  {q}")

Queries de share of voice: 4
Costo por ejecucion: $0.020
  @PalomaValenciaL lang:es -is:retweet
  @ABDELAESPRIELLA lang:es -is:retweet
  @JDOviedoAr lang:es -is:retweet
  @IvanCepedaCast lang:es -is:retweet


In [19]:
# Descomenta para ejecutar
today = date.today().isoformat()
sov_counts = th.run_count_recent(candidate_queries)
sov_counts.write_parquet(thanos_path(f"share_of_voice_{today}"))
print(sov_counts)

Ejecutando 4 queries count_recent (costo: $0.020)
shape: (32, 3)
┌──────────────────────────────────────┬────────────┬───────┐
│ query                                ┆ date       ┆ count │
│ ---                                  ┆ ---        ┆ ---   │
│ str                                  ┆ str        ┆ i64   │
╞══════════════════════════════════════╪════════════╪═══════╡
│ @PalomaValenciaL lang:es -is:retweet ┆ 2026-03-29 ┆ 782   │
│ @PalomaValenciaL lang:es -is:retweet ┆ 2026-03-30 ┆ 6285  │
│ @PalomaValenciaL lang:es -is:retweet ┆ 2026-03-31 ┆ 5473  │
│ @PalomaValenciaL lang:es -is:retweet ┆ 2026-04-01 ┆ 5704  │
│ @PalomaValenciaL lang:es -is:retweet ┆ 2026-04-02 ┆ 10157 │
│ @PalomaValenciaL lang:es -is:retweet ┆ 2026-04-03 ┆ 6399  │
│ @PalomaValenciaL lang:es -is:retweet ┆ 2026-04-04 ┆ 4513  │
│ @PalomaValenciaL lang:es -is:retweet ┆ 2026-04-05 ┆ 4767  │
│ @ABDELAESPRIELLA lang:es -is:retweet ┆ 2026-03-29 ┆ 738   │
│ @ABDELAESPRIELLA lang:es -is:retweet ┆ 2026-03-30 ┆ 7141  │
│ @AB

In [20]:
# Visualizacion: share of voice acumulado (si existen datos)
sov_files = sorted(_glob.glob(str(THANOS_DIR / "thanos_share_of_voice_*.parquet")))
if sov_files:
    all_sov = pl.concat([pl.read_parquet(f) for f in sov_files])
    fig = px.area(
        all_sov.to_pandas(),
        x="date", y="count", color="query",
        title="Share of voice por candidato (E10)",
        groupnorm="percent",
    )
    fig.show()
else:
    print("No hay datos de share of voice aun.")

## Fase 2: Descarga quirurgica (~$30-$80)

CADA estrategia ejecuta `count_recent` primero (T1) para estimar costo antes de descargar.
Las descargas estan comentadas por defecto.

### E11. Tweets cross-bloque

Tweets que mencionan candidatos de AMBOS bloques en el mismo tweet.
Son los mas valiosos: personas comparando candidatos activamente.

Estimado: 200-1,000 tweets -> $1-$5.

In [21]:
cross_query = th.build_cross_bloc_query(candidates)
print(f"Query cross-bloque:\n  {cross_query}")
print(f"\nLongitud: {len(cross_query)} chars")

# Descomenta para estimar volumen via count_recent (costo: $0.005)
cross_count = th.run_count_recent([cross_query])
total = cross_count["count"].sum()
print(f"\nVolumen estimado: {total:,} tweets")
print(f"Costo de descarga: ${total * 0.005:.2f}")

Query cross-bloque:
  (@PalomaValenciaL OR @ABDELAESPRIELLA OR @JDOviedoAr OR @CeDemocratico) (@IvanCepedaCast OR @PizarroMariaJo OR @GustavoBolivar OR @PactoHistorico) lang:es -is:retweet

Longitud: 166 chars
Ejecutando 1 queries count_recent (costo: $0.005)

Volumen estimado: 3,599 tweets
Costo de descarga: $18.00


In [22]:
# Descomenta para descargar (revisar costo arriba primero)
# cross_tweets = search_tweets(cross_query, max_pages=10)
# save_tweets(cross_tweets, "thanos_cross_bloc")
# print(f"Descargados: {len(cross_tweets)} tweets cross-bloque")

  p1: +99 (total: 99)
  p2: +99 (total: 198)
  p3: +99 (total: 297)
  p4: +100 (total: 397)
  p5: +99 (total: 496)
  p6: +99 (total: 595)
  p7: +97 (total: 692)
  p8: +90 (total: 782)
  p9: +89 (total: 871)
  p10: +87 (total: 958)
  Limite alcanzado: 10 paginas (958 tweets)
Saved 950 tweets to /mnt/storage/proyectos/entendiendo_dinamica_politica/notebooks/../data/raw/thanos_cross_bloc.parquet
Descargados: 958 tweets cross-bloque


### E12. Top N hilos de conversacion por reply_count

Ordenar tweets de candidatos por `reply_count` descendente (dato en disco, gratis).
Descargar solo los hilos con mas respuestas ciudadanas.

Estimado: 50 hilos x ~100 replies = 5,000 tweets -> $25.

In [27]:
top_threads = th.top_reply_threads(df_all, seed_handles, top_n=10)
print(f"Top 50 hilos por reply_count:")
print(top_threads)

total_replies = top_threads["reply_count"].sum()
print(f"\nTotal replies estimados: {total_replies:,}")
print(f"Costo estimado de descarga: ${total_replies * 0.005:.2f}")

Top 50 hilos por reply_count:
shape: (10, 7)
┌──────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬────────────┐
│ conversation ┆ tweet_id     ┆ author_user ┆ text        ┆ reply_count ┆ capped_repl ┆ like_count │
│ _id          ┆ ---          ┆ name        ┆ ---         ┆ ---         ┆ ies         ┆ ---        │
│ ---          ┆ str          ┆ ---         ┆ str         ┆ i64         ┆ ---         ┆ i64        │
│ str          ┆              ┆ str         ┆             ┆             ┆ i64         ┆            │
╞══════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪════════════╡
│ 203317561429 ┆ 203317561429 ┆ JDOviedoAr  ┆ Tal vez     ┆ 3435        ┆ 500         ┆ 13556      │
│ 5814395      ┆ 5814395      ┆             ┆ esta es la  ┆             ┆             ┆            │
│              ┆              ┆             ┆ primera vez ┆             ┆             ┆            │
│              ┆              ┆             ┆ 

In [24]:
# Descomenta para descargar los top N hilos
# N_HILOS = 20  # Ajustar segun presupuesto
# all_replies = []
# for row in top_threads.head(N_HILOS).iter_rows(named=True):
#      cid = row["conversation_id"]
#      query = f"conversation_id:{cid} lang:es"
#      replies = search_tweets(query, max_pages=5)
#      all_replies.extend(replies)
#      print(f"  Hilo {cid}: {len(replies)} replies")
#
# save_tweets(all_replies, "thanos_replies")
# print(f"\nTotal descargado: {len(all_replies)} replies")

### E13. Quote tweets de publicaciones virales

Las citas contienen opinion original + contexto del tweet citado.
Especialmente ricas para deteccion de sarcasmo.

Estimado: 1,000-4,000 citas -> $5-$20.

In [28]:
top_quoted = th.top_quoted_tweets(df_all, top_n=30)
print("Top 30 tweets por quote_count:")
print(top_quoted)

total_quotes = top_quoted["quote_count"].sum()
print(f"\nTotal quotes estimados: {total_quotes:,}")
print(f"Costo estimado: ${total_quotes * 0.005:.2f}")

Top 30 tweets por quote_count:
shape: (30, 5)
┌─────────────────────┬─────────────────┬───────────────────────────────┬─────────────┬────────────┐
│ tweet_id            ┆ author_username ┆ text                          ┆ quote_count ┆ like_count │
│ ---                 ┆ ---             ┆ ---                           ┆ ---         ┆ ---        │
│ str                 ┆ str             ┆ str                           ┆ i64         ┆ i64        │
╞═════════════════════╪═════════════════╪═══════════════════════════════╪═════════════╪════════════╡
│ 2033717031095153133 ┆ PalomaValenciaL ┆ Nos reunimos con el Gran      ┆ 747         ┆ 2556       │
│                     ┆                 ┆ Rabino David Michaan y varios ┆             ┆            │
│                     ┆                 ┆ miembros de la comunidad …    ┆             ┆            │
│ 2033175614295814395 ┆ JDOviedoAr      ┆ Tal vez esta es la primera    ┆ 536         ┆ 13556      │
│                     ┆                 ┆ vez

In [29]:
# Descomenta para descargar quotes
  # Descargar quotes usando endpoint dedicado (get_quoted)
all_quotes = th.download_quotes(top_quoted, max_quotes_per_tweet=100)
save_tweets(all_quotes, "thanos_quotes")
print(f"\nTotal: {len(all_quotes)} quotes descargados")

  p1: +99 (total: 99)
  Limite alcanzado: 1 paginas (99 tweets)
  Tweet 2033717031095153133 (PalomaValenciaL): 99 quotes
  p1: +100 (total: 100)
  Limite alcanzado: 1 paginas (100 tweets)
  Tweet 2033175614295814395 (JDOviedoAr): 100 quotes
  p1: +100 (total: 100)
  Limite alcanzado: 1 paginas (100 tweets)
  Tweet 2039710946885505252 (PalomaValenciaL): 100 quotes
  p1: +100 (total: 100)
  Limite alcanzado: 1 paginas (100 tweets)
  Tweet 2033873773556863473 (PalomaValenciaL): 100 quotes
  p1: +100 (total: 100)
  Limite alcanzado: 1 paginas (100 tweets)
  Tweet 2034684974750724358 (JDOviedoAr): 100 quotes
  p1: +99 (total: 99)
  Limite alcanzado: 1 paginas (99 tweets)
  Tweet 2033211391956386045 (PalomaValenciaL): 99 quotes
  p1: +100 (total: 100)
  Limite alcanzado: 1 paginas (100 tweets)
  Tweet 2039389034523324583 (ABDELAESPRIELLA): 100 quotes
  p1: +100 (total: 100)
  Limite alcanzado: 1 paginas (100 tweets)
  Tweet 2032477376664269030 (JDOviedoAr): 100 quotes
  p1: +100 (total: 100)

### E14. Menciones ciudadanas con engagement

Busqueda general de menciones a candidatos desde cuentas no-semilla,
con filtro de engagement progresivo (T3).

Estimado con min_faves:20 -> ~2,000-5,000 tweets -> $10-$25.

In [30]:
  # E14: Menciones ciudadanas con engagement
  # min_faves NO es compatible con counts/recent, solo con search_recent/all.
  # Estimamos volumen total sin filtro; min_faves se aplica al descargar con

  candidate_handles = " OR ".join(
      f"@{a['handle'].lstrip('@')}"
      for accounts in candidates.values()
      for a in accounts
      if a.get("rol", "").startswith("Candidat")
  )
  base_query = f"({candidate_handles}) lang:es -is:retweet"

  # Volumen total (sin min_faves) -- costo: $0.005
  counts = th.run_count_recent([base_query])
  total = counts["count"].sum()
  print(f"Menciones totales a candidatos (7d): {total:,} tweets")
  print(f"Costo descarga sin filtro: ${total * 0.005:.2f}")
  print(f"\nPara descargar con filtro, usar search_recent con min_faves:")
  for min_faves in (100, 50, 20):
      print(f"  {base_query} min_faves:{min_faves}")

Ejecutando 1 queries count_recent (costo: $0.005)
Menciones totales a candidatos (7d): 141,293 tweets
Costo descarga sin filtro: $706.47

Para descargar con filtro, usar search_recent con min_faves:
  (@PalomaValenciaL OR @ABDELAESPRIELLA OR @JDOviedoAr OR @IvanCepedaCast) lang:es -is:retweet min_faves:100
  (@PalomaValenciaL OR @ABDELAESPRIELLA OR @JDOviedoAr OR @IvanCepedaCast) lang:es -is:retweet min_faves:50
  (@PalomaValenciaL OR @ABDELAESPRIELLA OR @JDOviedoAr OR @IvanCepedaCast) lang:es -is:retweet min_faves:20


In [31]:
# Descomenta para descargar el tier seleccionado
citizen_mentions = th.download_citizen_mentions(candidates, min_likes=50, max_pages=20)
save_tweets(citizen_mentions, "thanos_citizen_mentions")
print(f"Descargados: {len(citizen_mentions)} menciones ciudadanas")

Query: (@PalomaValenciaL OR @ABDELAESPRIELLA OR @JDOviedoAr OR @IvanCepedaCast) lang:es -is:retweet min_lik...
  p1: +100 (total: 100)
  p2: +100 (total: 200)
  p3: +100 (total: 300)
  p4: +100 (total: 400)
  p5: +99 (total: 499)
  p6: +100 (total: 599)
  p7: +99 (total: 698)
  p8: +100 (total: 798)
  p9: +100 (total: 898)
  p10: +100 (total: 998)
  p11: +99 (total: 1097)
  p12: +100 (total: 1197)
  p13: +100 (total: 1297)
  p14: +100 (total: 1397)
  p15: +99 (total: 1496)
  p16: +99 (total: 1595)
  p17: +98 (total: 1693)
  p18: +100 (total: 1793)
  p19: +100 (total: 1893)
  p20: +100 (total: 1993)
  Limite alcanzado: 20 paginas (1993 tweets)
Descargados: 1993 tweets (min_likes:50)
Costo: $9.96
Saved 1974 tweets to /mnt/storage/proyectos/entendiendo_dinamica_politica/notebooks/../data/raw/thanos_citizen_mentions.parquet
Descargados: 1993 menciones ciudadanas


### E15. Recoleccion alineada a fechas de encuestas

Descargar tweets en ventanas de +/- 3 dias alrededor de cada encuesta.
Maximiza la correlacion tweet-encuesta que THANOS necesita para calibrar.

Estimado: ~10 encuestas x 1,000 tweets = 10,000 tweets -> $50.

In [32]:
windows = th.poll_date_windows(encuestas, window_days=3)

print(f"Ventanas de encuestas ({len(windows)}):")
for w in windows:
    print(f"  {w['encuestadora']}: {w['fecha_ref']} -> [{w['start'][:10]}, {w['end'][:10]}]")

total_est = len(windows) * 1000  # estimacion conservadora
print(f"\nEstimacion conservadora: {total_est:,} tweets -> ${total_est * 0.005:.2f}")

Ventanas de encuestas (7):
  Guarumo/EcoAnalítica: 2026-03-25 -> [2026-03-22, 2026-03-28]
  CNC: 2026-03-21 -> [2026-03-18, 2026-03-24]
  GAD3: 2026-03-18 -> [2026-03-15, 2026-03-21]
  AtlasIntel: 2026-03-12 -> [2026-03-09, 2026-03-15]
  AtlasIntel: 2026-03-12 -> [2026-03-09, 2026-03-15]
  AtlasIntel: 2026-03-12 -> [2026-03-09, 2026-03-15]
  AtlasIntel: 2026-03-12 -> [2026-03-09, 2026-03-15]

Estimacion conservadora: 7,000 tweets -> $35.00


In [ ]:
# Descomenta para descargar tweets alineados a encuestas
# all_poll_tweets = []
# for w in windows:
#     query = f"({candidate_handles}) lang:es -is:retweet"
#     tweets = search_tweets(query, start_time=w["start"], max_pages=10)
#     all_poll_tweets.extend(tweets)
#     print(f"  {w['encuestadora']} ({w['fecha_ref']}): {len(tweets)} tweets")
#
# save_tweets(all_poll_tweets, "thanos_poll_aligned")
# print(f"\nTotal: {len(all_poll_tweets)} tweets alineados a encuestas")

### Resumen Fase 2

Despues de ejecutar las descargas, verificar los datasets generados.

In [33]:
# Verificar datasets de fase 2 (si existen)
fase_2_datasets = ["thanos_cross_bloc", "thanos_replies", "thanos_quotes",
                   "thanos_citizen_mentions", "thanos_poll_aligned"]
for name in fase_2_datasets:
    path = DATA_RAW / f"{name}.parquet"
    if path.exists():
        df_tmp = pl.read_parquet(path)
        print(f"  {name}: {len(df_tmp):,} tweets")
    else:
        print(f"  {name}: (pendiente)")

  thanos_cross_bloc: 950 tweets
  thanos_replies: 874 tweets
  thanos_quotes: 2,994 tweets
  thanos_citizen_mentions: 1,974 tweets
  thanos_poll_aligned: (pendiente)


## Fase 3: Expansion de la red via listas curadas (~$50-$150)

En lugar de expansion snowball algoritmica, se usan listas curadas en X:
el usuario crea listas privadas agrupando cuentas por rol, y el endpoint
`/2/lists/:id/tweets` recolecta todos los tweets en una sola llamada paginada.

Listas propuestas:

| Lista | Contenido | Cuentas estimadas |
|---|---|---|
| Medios | ElTiempo, Semana, ElEspectador, RCN, Caracol, etc. | ~15 |
| Periodistas/analistas | Vicky Davila, Daniel Coronell, Salud Hernandez, Nestor Morales, etc. | ~15 |
| Influencers derecha | Cuentas con alto in_degree mencionadas por semillas de derecha | ~20 |
| Influencers izquierda | Cuentas con alto in_degree mencionadas por semillas de izquierda | ~20 |
| Politicos clave no-candidatos | Congresistas, gobernadores, alcaldes que opinan activamente | ~20 |

Las cuentas se seleccionan a partir de los nodos organicos de Fase 0 (in_degree)
y las menciones/RT descubiertos en Fase 2.

### E16. Listas curadas por categoria

Crear listas privadas en X con cuentas seleccionadas a partir de los nodos organicos
descubiertos en Fase 0 y las menciones/RT de Fase 2. Aplicar T4: una sola llamada
por lista via `/2/lists/:id/tweets`.

Estimado: 90 cuentas x ~300 tweets = 27,000 tweets -> $135.

In [49]:
# Nodos organicos como candidatos para listas curadas
in_degree = pl.read_parquet(DATA_PROCESSED / "interaction_in_degree.parquet")
organic_top = in_degree.filter(~pl.col("is_seed")).head(50)

print("Top 30 nodos organicos (candidatos para listas curadas):")
print(organic_top)

# Agrupar manualmente en listas:
# - Medios: eltiempo, semaborja, etc.
# - Periodistas: VickyDavilaH, DCoronell, etc.
# - Influencers derecha/izquierda: segun bloque de quien los menciona
# - Politicos clave: congresistas, gobernadores activos

Top 30 nodos organicos (candidatos para listas curadas):
shape: (50, 7)
┌─────────────────┬───────────┬─────────────┬───────┬────────────┬────────────────┬─────────┐
│ target          ┆ in_degree ┆ mentions_in ┆ rt_in ┆ replies_in ┆ unique_sources ┆ is_seed │
│ ---             ┆ ---       ┆ ---         ┆ ---   ┆ ---        ┆ ---            ┆ ---     │
│ str             ┆ u32       ┆ u32         ┆ u32   ┆ u32        ┆ u32            ┆ bool    │
╞═════════════════╪═══════════╪═════════════╪═══════╪════════════╪════════════════╪═════════╡
│ aida_quilcue    ┆ 53        ┆ 53          ┆ 0     ┆ 0          ┆ 6              ┆ false   │
│ revistasemana   ┆ 30        ┆ 30          ┆ 0     ┆ 0          ┆ 7              ┆ false   │
│ jrestrp         ┆ 27        ┆ 27          ┆ 0     ┆ 0          ┆ 1              ┆ false   │
│ diegoasantos    ┆ 19        ┆ 19          ┆ 0     ┆ 0          ┆ 2              ┆ false   │
│ juanmanuelgalan ┆ 16        ┆ 16          ┆ 0     ┆ 0          ┆ 4              

In [52]:
edges.join(organic_top, how="inner", on="target").group_by(["source", "target"]).len().sort(["target", "len"]).write_csv("../docs/conteo_segidores.txt")

In [ ]:
# Descomenta para recolectar tweets via listas de X
# Las listas se crean manualmente en la UI de Twitter.
#
# LIST_IDS = {
#     "medios": "LIST_ID_MEDIOS",
#     "periodistas": "LIST_ID_PERIODISTAS",
#     "influencers_derecha": "LIST_ID_INF_DER",
#     "influencers_izquierda": "LIST_ID_INF_IZQ",
#     "politicos_clave": "LIST_ID_POL",
# }
#
# client = get_client()
# all_list_tweets = []
# for list_name, list_id in LIST_IDS.items():
#     print(f"Recolectando lista: {list_name}...")
#     # TODO: implementar collect_list_tweets en tweet_collector.py
#     pass
#
# save_tweets(all_list_tweets, "thanos_curated_lists")

### E17. Identificacion de swing users via replies bidireccional

Usuarios que respondieron a candidatos de AMBOS bloques.
Estos son votantes indecisos activos -- la senal mas directa
para predecir movimiento de votos.

Requiere datos de Fase 2 (E12 replies) para identificarlos.

In [ ]:
# Identificar swing users (requiere datos de fase 2)
replies_path = DATA_RAW / "thanos_replies.parquet"
if replies_path.exists():
    citizen_tweets = pl.read_parquet(replies_path)
    swing = th.identify_swing_users(citizen_tweets, candidates)
    swing.write_parquet(thanos_path("swing_users"))
    print(f"Swing users encontrados: {len(swing)}")
    print(swing.head(20))
else:
    print("Ejecutar E12 primero para obtener replies ciudadanos")

In [ ]:
# Descomenta para descargar timelines de swing users
# if thanos_path("swing_users").exists():
#     swing = pl.read_parquet(thanos_path("swing_users"))
#     top_swing = swing.head(30)
#     swing_records = []
#     for row in top_swing.iter_rows(named=True):
#         username = row["author_username"]
#         user_id = resolve_user_ids([username]).get(username)
#         if user_id:
#             records = collect_user_timeline(user_id)
#             swing_records.extend(records)
#     save_tweets(swing_records, "thanos_swing_timelines")

### Resumen Fase 3

In [ ]:
# Estado de expansion de red
fase_3_datasets = ["thanos_curated_lists", "thanos_swing_timelines"]
for name in fase_3_datasets:
    path = DATA_RAW / f"{name}.parquet"
    if path.exists():
        df_tmp = pl.read_parquet(path)
        print(f"  {name}: {len(df_tmp):,} tweets")
    else:
        print(f"  {name}: PENDIENTE")

## Fase 1b: Monitoreo de hashtags con diccionario enriquecido (~$3-$7/mes)

Se activa despues de las Fases 2 y 3. El diccionario de hashtags ahora incluye:
- **Campana:** hashtags creados por las campanas oficiales (ej. #PalomaPresidenta)
- **Organicos:** hashtags emergentes de la ciudadania (ej. #ColombiaDespierta)
- **Coyuntura:** hashtags ligados a temas del momento (ej. #ReformaLaboral)

El diccionario se enriquece con hashtags descubiertos en Fases 2-3,
no solo los de propaganda de las semillas.

In [ ]:
# E8: Tracking diario de hashtags via count_recent (diccionario enriquecido)
hashtag_dict = pl.read_parquet(thanos_path("hashtag_dict"))

# Enriquecer con hashtags de tweets ciudadanos (si existen)
for name in ["thanos_cross_bloc", "thanos_replies", "thanos_quotes",
             "thanos_citizen_mentions", "thanos_curated_lists"]:
    path = DATA_RAW / f"{name}.parquet"
    if path.exists():
        df_extra = pl.read_parquet(path)
        extra_ht = th.hashtags_by_bloc(df_extra)
        hashtag_dict = pl.concat([hashtag_dict, extra_ht]).group_by("hashtag", "bloque").agg(
            pl.col("count").sum()
        ).sort("count", descending=True)

enriched_queries = th.build_hashtag_queries(hashtag_dict, top_n=50)

print(f"Queries enriquecidas: {len(enriched_queries)} hashtags")
print(f"Costo por ejecucion: ${len(enriched_queries) * 0.005:.3f}")
for q in enriched_queries[:10]:
    print(f"  {q}")
if len(enriched_queries) > 10:
    print(f"  ... y {len(enriched_queries) - 10} mas")

## Ensamblaje del Modelo THANOS

Cargar todas las variables computadas en Fases 0, 1a, 2, 3, 1b y construir la matriz de regresion.

### Variables del modelo

| Variable | Formula | Fuente |
|---|---|---|
| `x_{t,h,l}^(k)` | Proporcion de top 10 hashtags del partido k en ventana (t-h, t-l) | E1 + E8 (Fase 1b, diccionario enriquecido) |
| `h_t` | Centralidad armonica del nodo mas influyente | E16 listas curadas (Fase 3) |
| `r_t^(k)` | Proporcion de retweets del influenciador principal del partido k | E2 + datos expandidos |
| `y_t` | Proporcion de votos segun encuestas | `poll_scraper.py` |

### Calculo de h_t: Centralidad armonica

In [ ]:
# Construir grafo desde todas las edges disponibles
all_edges = [edges]  # Edges originales

# Agregar edges de expansion si existen (listas curadas)
for name in ["thanos_curated_lists", "thanos_swing_timelines"]:
    path = DATA_RAW / f"{name}.parquet"
    if path.exists():
        df_exp = pl.read_parquet(path)
        new_edges = th.extract_edges(df_exp)
        all_edges.append(new_edges)
        print(f"  + {len(new_edges):,} edges desde {name}")

combined_edges = pl.concat(all_edges).unique()
print(f"Total edges: {len(combined_edges):,}")

# Construir grafo NetworkX
G = nx.DiGraph()
for row in combined_edges.iter_rows(named=True):
    G.add_edge(row["source"], row["target"], edge_type=row["edge_type"])

# Centralidad armonica
h_t = nx.harmonic_centrality(G)
top_central = sorted(h_t.items(), key=lambda x: x[1], reverse=True)[:20]
print(f"\nTop 20 por centralidad armonica:")
for user, score in top_central:
    print(f"  {user}: {score:.4f}")

### Calculo de r_t^(k): Proporcion de retweets

In [ ]:
# Usar datos expandidos si estan disponibles, si no usar semilla
all_tweets = df_all
for name in ["thanos_curated_lists", "thanos_swing_timelines", "thanos_replies"]:
    path = DATA_RAW / f"{name}.parquet"
    if path.exists():
        df_exp = pl.read_parquet(path)
        if "bloque" not in df_exp.columns:
            df_exp = th.assign_bloque(df_exp, candidates)
        all_tweets = pl.concat([all_tweets, df_exp]).unique(subset=["tweet_id"])

rt_expanded = th.retweet_chains(all_tweets, candidates)
print(f"r_t^(k) con datos expandidos ({len(all_tweets):,} tweets):")
print(rt_expanded)

### Calculo de x_{t,h,l}^(k): Proporcion de hashtags

Se calcula para multiples ventanas (h, l) segun el modelo THANOS.
Si no hay datos de count_recent (E8), se usa la frecuencia relativa de los datos en disco.

In [ ]:
# Grid de ventanas (h, l) en dias
WINDOWS = [(7, 1), (14, 1), (14, 7), (21, 1), (21, 7), (21, 14), (28, 1)]

# Verificar si hay datos de count_recent
count_files = sorted(_glob.glob(str(THANOS_DIR / "thanos_hashtag_counts_*.parquet")))

if count_files:
    # Usar datos reales de count_recent
    all_counts = pl.concat([pl.read_parquet(f) for f in count_files])
    ref_date = all_counts["date"].max()
    print(f"Usando datos de count_recent (fecha ref: {ref_date})")
    print(f"Ventanas (h, l): {WINDOWS}")

    x_values = {}
    for h, l in WINDOWS:
        props = th.compute_hashtag_proportions(all_counts, hashtag_dict, h, l, ref_date)
        x_values[(h, l)] = props
        print(f"  ({h:2d}, {l:2d}): {props}")
else:
    print("Sin datos de count_recent. Calculando proporcion estatica desde datos en disco.")
    print("(Ejecutar E8 para obtener series temporales reales)")

    # Proporcion estatica: count de hashtags del bloque / total hashtags
    total_ht = hashtag_dict["count"].sum()
    x_values = {}
    for h, l in WINDOWS:
        props = {}
        for bloque in ("derecha", "izquierda"):
            top_10 = hashtag_dict.filter(pl.col("bloque") == bloque).head(10)
            bloc_count = top_10["count"].sum()
            props[f"x_{bloque}"] = bloc_count / total_ht if total_ht > 0 else 0.0
        x_values[(h, l)] = props
        print(f"  ({h:2d}, {l:2d}): {props}")

### Preparacion de y_t: Datos de encuestas

In [ ]:
# Preparar y_t para el candidato de interes
# THANOS modela un candidato a la vez; el mas relevante es el que tiene mas datos
print("Columnas de candidatos disponibles:")
candidate_cols = [c for c in encuestas.columns if c not in (
    "encuestadora", "fecha", "muestra", "otros", "blanco", "ninguno",
    "ns_nr", "margen_error", "fetched_at", "fecha_ref",
)]
for col in candidate_cols:
    non_null = encuestas[col].drop_nulls().len()
    print(f"  {col}: {non_null} observaciones")

# Usar Cepeda PH como target (ajustar segun necesidad)
TARGET_COL = "Cepeda PH"
y_df = th.prepare_y_t(encuestas, target_col=TARGET_COL)
print(f"\ny_t para {TARGET_COL}:")
print(y_df)

### Ensamblaje de la feature matrix

In [ ]:
# Construir una fila por encuesta con todas las features
# NOTA: con datos estaticos (sin count_recent), x es igual para todas las ventanas.
# Con datos reales de E8, cada ventana produce un x diferente.

# Por ahora: usar la primera ventana como representativa
h_ref, l_ref = WINDOWS[0]
x_ref = x_values[(h_ref, l_ref)]

rows = []
for row in y_df.iter_rows(named=True):
    rows.append({
        "encuestadora": row.get("encuestadora", ""),
        "y_t": row["y_t"],
        "log_odds_y": row["log_odds_y"],
        "x_1": x_ref.get("x_derecha", 0),
        "x_2": x_ref.get("x_izquierda", 0),
        "h_t": h_t,
        "r_1": rt_proportions.get("r_derecha", 0),
        "r_2": rt_proportions.get("r_izquierda", 0),
    })

feature_matrix = pl.DataFrame(rows)
feature_matrix = feature_matrix.with_columns(
    (pl.col("x_1") * pl.col("x_2")).alias("x_1_x_2")
)

print("Feature matrix:")
print(feature_matrix)
feature_matrix.write_parquet(thanos_path("feature_matrix"))

### Modelo THOS (solo hashtags)

Ecuacion 3.1 del paper:
`logit(y_t) = b0 + b1*x_1 + b2*x_2 + b3*x_1*x_2`

Modelo mas simple con 4 parametros. Viable con pocas observaciones.

In [ ]:
# Ajustar THOS
thos_cols = ["x_1", "x_2", "x_1_x_2"]
print(f"Observaciones: {len(feature_matrix)}")
print(f"Parametros THOS: {len(thos_cols) + 1} (incl. constante)")

if len(feature_matrix) > len(thos_cols) + 1:
    thos_result = th.fit_thanos(feature_matrix, y_col="y_t", x_cols=thos_cols)
    print(thos_result.summary())
else:
    print(f"\nADVERTENCIA: {len(feature_matrix)} observaciones < {len(thos_cols) + 1} parametros")
    print("El modelo esta sobredeterminado. Los resultados no son confiables.")
    print("Se necesitan mas encuestas para un ajuste valido.")
    print("Ajustando de todas formas para exploracion...")
    try:
        thos_result = th.fit_thanos(feature_matrix, y_col="y_t", x_cols=thos_cols)
        print(thos_result.summary())
    except Exception as e:
        print(f"Error en ajuste: {e}")
        thos_result = None

### Modelo THANOS completo

Ecuacion 3.2 del paper:
`logit(y_t) = b0 + b1*x_1 + b2*x_2 + b3*x_1*x_2 + b4*h_t + b5*r_1 + b6*r_2`

7 parametros. Con solo 7 encuestas el sobreajuste es casi seguro.

In [ ]:
thanos_cols = ["x_1", "x_2", "x_1_x_2", "h_t", "r_1", "r_2"]
print(f"Observaciones: {len(feature_matrix)}")
print(f"Parametros THANOS: {len(thanos_cols) + 1} (incl. constante)")

if len(feature_matrix) > len(thanos_cols) + 1:
    thanos_result = th.fit_thanos(feature_matrix, y_col="y_t", x_cols=thanos_cols)
    print(thanos_result.summary())
else:
    print(f"\nADVERTENCIA: {len(feature_matrix)} observaciones <= {len(thanos_cols) + 1} parametros")
    print("Sobreajuste garantizado. Solo para exploracion.")
    try:
        thanos_result = th.fit_thanos(feature_matrix, y_col="y_t", x_cols=thanos_cols)
        print(thanos_result.summary())
    except Exception as e:
        print(f"Error: {e}")
        thanos_result = None

### Comparacion THOS vs THANOS

In [ ]:
if thos_result is not None and thanos_result is not None:
    print("Comparacion de modelos:")
    print(f"{'Metrica':<20} {'THOS':>12} {'THANOS':>12}")
    print("-" * 46)
    print(f"{'AIC':<20} {thos_result.aic:>12.4f} {thanos_result.aic:>12.4f}")
    print(f"{'BIC':<20} {thos_result.bic:>12.4f} {thanos_result.bic:>12.4f}")
    print(f"{'Deviance':<20} {thos_result.deviance:>12.4f} {thanos_result.deviance:>12.4f}")
    print(f"{'Parametros':<20} {thos_result.df_model + 1:>12.0f} {thanos_result.df_model + 1:>12.0f}")
    print(f"{'Obs':<20} {thos_result.nobs:>12.0f} {thanos_result.nobs:>12.0f}")

    # Predicciones
    print("\nPredicciones:")
    thos_pred = thos_result.predict()
    thanos_pred = thanos_result.predict()
    y_actual = feature_matrix["y_t"].to_numpy()

    for i in range(len(y_actual)):
        enc = feature_matrix["encuestadora"][i] if "encuestadora" in feature_matrix.columns else f"obs_{i}"
        print(f"  {enc}: actual={y_actual[i]:.3f}  THOS={thos_pred[i]:.3f}  THANOS={thanos_pred[i]:.3f}")
elif thos_result is not None:
    print("Solo THOS disponible (THANOS no convergio)")
    print(f"AIC: {thos_result.aic:.4f}")
    print(f"BIC: {thos_result.bic:.4f}")
else:
    print("Ningun modelo convergio. Se necesitan mas datos.")

### Prediccion final

In [ ]:
# Promedio de predicciones sobre ventanas
if thos_result is not None:
    window_predictions = {}
    for (h, l), x_vals in x_values.items():
        X_new = np.array([[
            1.0,  # constante
            x_vals.get("x_derecha", 0),
            x_vals.get("x_izquierda", 0),
            x_vals.get("x_derecha", 0) * x_vals.get("x_izquierda", 0),
        ]])
        pred = thos_result.predict(X_new)[0]
        window_predictions[(h, l)] = pred
        print(f"  Ventana ({h:2d}, {l:2d}): p_hat = {pred:.4f} ({pred*100:.1f}%)")

    p_final = th.average_window_predictions(window_predictions)
    print(f"\n{'='*50}")
    print(f"Prediccion THOS para {TARGET_COL}: {p_final:.4f} ({p_final*100:.1f}%)")
    print(f"{'='*50}")

    # Guardar predicciones
    pred_df = pl.DataFrame({
        "modelo": ["THOS"],
        "candidato": [TARGET_COL],
        "prediccion": [p_final],
        "n_ventanas": [len(window_predictions)],
        "n_observaciones": [int(thos_result.nobs)],
        "aic": [thos_result.aic],
    })
    pred_df.write_parquet(thanos_path("predictions"))
    print(f"\nGuardado en: thanos_predictions.parquet")
else:
    print("No hay modelo ajustado. Ejecutar las fases anteriores primero.")

## Diagnosticos del modelo

Analisis detallado de los resultados estadisticos.

In [ ]:
# Summary completo del mejor modelo disponible
best_model = thanos_result if thanos_result is not None else thos_result
model_name = "THANOS" if thanos_result is not None else "THOS"

if best_model is not None:
    print(f"=== Diagnosticos {model_name} ===\n")
    print(best_model.summary())
    print(f"\n--- Coeficientes con intervalos de confianza ---")
    print(best_model.summary2().tables[1])
else:
    print("No hay modelo ajustado.")

In [ ]:
# Deviance residuals y QQ plot
if best_model is not None:
    residuals = best_model.resid_deviance
    fitted = best_model.predict()

    # Residuals vs Fitted
    fig = px.scatter(
        x=fitted, y=residuals,
        labels={"x": "Valores ajustados", "y": "Deviance residuals"},
        title=f"Residuals vs Fitted ({model_name})",
    )
    fig.add_hline(y=0, line_dash="dash", line_color="red")
    fig.show()

    # QQ plot
    from scipy import stats
    sorted_resid = np.sort(residuals)
    theoretical = stats.norm.ppf(np.linspace(0.01, 0.99, len(sorted_resid)))

    fig = px.scatter(
        x=theoretical, y=sorted_resid,
        labels={"x": "Cuantiles teoricos", "y": "Cuantiles observados"},
        title=f"QQ Plot de residuales ({model_name})",
    )
    # Linea de referencia
    fig.add_trace(go.Scatter(
        x=[theoretical.min(), theoretical.max()],
        y=[theoretical.min(), theoretical.max()],
        mode="lines", line=dict(dash="dash", color="red"),
        showlegend=False,
    ))
    fig.show()

### Limitaciones y advertencias

1. **Pocas observaciones**: Con solo 7 encuestas, el modelo tiene muy pocos grados de libertad.
   THOS (4 params) es el limite practico; THANOS (7 params) sobreajusta garantizado.

2. **Red sesgada**: Sin ejecutar Fases 2-3, la red solo refleja el ecosistema interno
   de cada bloque. Las listas curadas (Fase 3) resuelven esto parcialmente.

3. **Hashtags de propaganda vs organicos**: Sin Fase 1b (post-enriquecimiento),
   el tracking de hashtags solo mide esfuerzo de campana, no opinion ciudadana.

4. **Serie temporal corta**: E9/E10 (Fase 1a) se deben ejecutar diariamente desde el inicio
   para maximizar datos disponibles antes de las elecciones (31 mayo 2026).

## Exploracion libre

Celdas para consultas ad-hoc sobre los datos procesados.

In [ ]:
# Ejemplo: explorar un hashtag especifico
# hashtag_dict.filter(pl.col("hashtag").str.contains("paloma"))

In [ ]:
# Ejemplo: ver la feature matrix completa
# pl.read_parquet(thanos_path("feature_matrix"))